# Geocoding Notes

### New York City

We use [NYC's Geosupport](https://www.nyc.gov/site/planning/data-maps/open-data/dwn-gde-home.page) via python bindings provided by [python-geosupport](https://github.com/ishiland/python-geosupport). 

Since it runs from desktop it can easily run through the whole database in a few minutes depending on the machine.

It also returns everything we need (bin, bbl, community districts, census tracts, council districts, and status messages) without the need to spatial join.

### Testing geocoding for NYState Addresses.

#### Option 1

[NYS ITS Geospatial Services](https://gis.ny.gov/address-geocoder) seems to provide the most accurate service. You can access the service by using an [ESRI API endpoint](https://gisservices.its.ny.gov/arcgis/rest/services/Locators/Street_and_Address_Composite/GeocodeServer/findAddressCandidates?Street=1+DAVENPORT+AVENUE&City=&State=&ZIP=10805&SingleLine=&outFields=*&maxLocations=&matchOutOfRange=true&langCode=&locationType=&sourceCountry=&category=&location=&distance=&searchExtent=&outSR=4326&magicKey=&f=html) and feed in `Street:` and `Zipcode:` to return a latitude and longitude. 

The downsides are the possible rate limits of the API (how much you query in a time internal), the half of million addresses we need to geocode, and returns only giving us lat/lng. To use this service we need to cache previous requests to reduce on recalling already geocoded addresses and spatial join to get geographic boundries like census tracts and elected districts.

#### Option 2

[The US Census Geocoder](https://geocoding.geo.census.gov/geocoder/) provides both batch (limit of 10k rows) and [single request APIs](https://geocoding.geo.census.gov/geocoder/locations/onelineaddress?address=1%20DAVENPORT%20AVENUE%2C%2010805&benchmark=4). We can also query [geographies](https://geocoding.geo.census.gov/geocoder/geographies/onelineaddress?address=4600+silver+hill+rd%2C+20233&benchmark=Public_AR_Census2020&vintage=Census2010_Census2020&format=json). There is also [a python wrapper](https://github.com/fitnr/censusgeocode).



--
#### Other Options

PostGIS with the US Census's Tiger has similar functionality but needs to be self hosted using docker.

[Pelias](https://github.com/pelias/docker/) is another alternative that we can host on docker but requires a preprocessed file of NYState "at least 8GB RAM," accuracy is lower, still requires a join. Justlab seems to have created a version for NYState on their [github](https://github.com/justlab/geocode_sparcs).

In [ ]:
import pandas as pd
import usaddress
import requests

In [ ]:
def parse_address(addr):
    """parses full address string and returns dict of address components needed for geocoding

    Args:
        addr (str): full street address

    Returns:
        dict: dict of strings for house_number, street_name, borough_code, and place_name
    """
    try:
        tags, _ = usaddress.tag(addr)
        house_number = ' '.join([v for k, v in tags.items() if k.startswith('AddressNumber')])
        street_name = ' '.join([v for k, v in tags.items() if k.startswith('StreetName')])
    except usaddress.RepeatedLabelError as e :
        tags = e.parsed_string
        house_number = ' '.join([x[0] for x in tags if x[1].startswith('AddressNumber')])
        street_name = ' '.join([x[0] for x in tags if x[1].startswith('StreetName')])
    return dict(
        house_number = house_number,
        street_name = street_name
    )

In [ ]:
addresses = pd.read_csv('./oca-addresses.csv')

### Test limits of NYS ITS

Takes about 1h 54s for 10k requests so about 45 hours for the full dataset. Does not seem to have rate limits but we do want to respect the api.

Could probably speed it up with parallel requests / multiprocessing. OR only use it as a fallback. There is also some slow down caused by usaddresses parase.

In [ ]:
def geocodeNYStateAddr(row):
    street1 = row['street1']
    postalcode = row['postalcode']
    
    if pd.isna(street1) or pd.isna(postalcode):
        return pd.Series([None, None, None]) 
    
    standard_street = parse_address(street1)
    
    params={
        'Street': standard_street['house_number'] + ' ' + standard_street['street_name'],
        'ZIP':postalcode,
        "matchOutOfRange": "false",
        "maxLocations": 1,
        "outSR":4326,
        "f": "json"
    }
    

    
    url = "https://gisservices.its.ny.gov/arcgis/rest/services/Locators/Street_and_Address_Composite/GeocodeServer/findAddressCandidates"
    try:
        r = requests.post(url, params = params)
        d = r.json()
    except:
        print('error \n',d)
  

    if 'candidates'in d and len(d['candidates']) > 0:
        addr = d['candidates'][0]['address']
        location = d['candidates'][0]['location']
        x = location['x']
        y = location['y']
        return pd.Series([addr, x, y])
    else:
        return pd.Series([None, None, None]) 

In [ ]:
%timeit -n 1 -r 1 addresses[:10000].apply(geocodeNYStateAddr, axis = 1)

### Code for batch census geocoding

1min 17s for 1000 with geographies and 32.3 s without, so about **10 minutes for 10k with geographies, and 5 minutes without.** ~12 times faster than the NYS GIS geocoder. 

4 hours for the full dataset. Can we can speed it up by running on four threads?

In [ ]:
#!pip install censusgeocode

In [ ]:
import censusgeocode as cg

In [ ]:
#format addresses and export to the format below without HEADERS
#unique id, street address, state, city, zip code

def formatStreetAddress(street1):
    if pd.isna(street1):
        return ''
    
    norm_street = parse_address(street1)
    return norm_street['house_number'] + ' ' + norm_street['street_name']

addresses['norm_street'] = addresses['street1'].apply(formatStreetAddress)

#use the pandas index as the id, AVOID using the indexnumberid has this data is sent to an external service.
addresses['id'] = addresses.index + 1

In [ ]:
addresses['null'] = None

In [ ]:
addresses[['id','norm_street','null','null','postalcode']].sample(1000).to_csv('./data/batch.csv', index = False, header= False)

In [ ]:
%timeit -n 1 -r 1 cg.addressbatch('./data/batch.csv', returntype='geographies')

In [ ]:
%timeit -n 1 -r 1 cg.addressbatch('./data/batch.csv', returntype='locations')

In [ ]:
df = pd.DataFrame(cg.addressbatch('./data/batch.csv', returntype='locations'))

In [ ]:
df['match'].value_counts()

In [ ]:
df['matchtype'].value_counts()

In [ ]:
df[df['matchtype'] == 'Non_Exact']

In [ ]:
df[pd.isna(df['matchtype'])]

In [ ]:
df

In [ ]:
#for each 10k entries



### SQL to get standardized GEOIDs for NYC and NYS census tracks.

PLUTO provides a `ct` column six-digit census tract number. ie `Census Tract 062600`. You would need the borough code to create a look. For example if the boro code for `Census Tract 062600` is 4, it will be queens county so we can prefix `36081`, `36` is the FIPS code for New York State and `36081` the [FIPS county code](https://en.wikipedia.org/wiki/List_of_United_States_FIPS_codes_by_county) for Queens .

| FIPS   | County Name     | Borough Name  | Boro Code |
|--------|-----------------|---------------|-----------|
| 36061  | New York County | Manhattan     | 1         |
| 36005  | Bronx County    | The Bronx     | 2         |
| 06031  | Kings County    | Brooklyn      | 3         |
| 36081  | Queens County   | Queens        | 4         |
| 36085  | Richmond County | Staten Island | 5         |

Census Batch Geocoder provides a GEOID `24033802405` if you choose to use returntype='geographies'.


```



```

---

Alternatively with a census tract of NY State in Postgres/gis you can use a spatial join. https://www.census.gov/cgi-bin/geo/shapefiles/index.php?year=2022&layergroup=Census+Tracts


```


```